In [1]:
from pathlib import Path
import os,json,hashlib,time,random
os.environ['CUBLAS_WORKSPACE_CONFIG']=':4096:8'
import numpy as np,h5py,torch
from torch import nn
from torch.utils.data import TensorDataset,DataLoader
from sklearn.metrics import accuracy_score,precision_score,recall_score,f1_score,roc_auc_score,average_precision_score,confusion_matrix,mean_absolute_error,mean_squared_error,r2_score
ROOT=Path(r'C:/Users/choud/Desktop/Minor project');OUT=ROOT/'bilstm_eew/bilstm_training_v1';DATA=ROOT/'bilstm_eew/causal_tabular_v1/tabular_features.h5'
CNN=ROOT/'cnn/cnn wave/eew_causal_v3/cnn_eew_training/codex_performance_audit/cnn_performance_audit_results.json'
VALIDATION=ROOT/'bilstm_eew/causal_tabular_v1/feature_validation_report.md'
CFG=dict(seed=42,hidden=32,layers=2,dropout=.2,batch_size=256,max_epochs=35,patience=7,min_delta=1e-4,lr=.001,weight_decay=.0001,clip=5.,magnitude_weight=1.,huber_delta=1.)
DEV=torch.device('cuda' if torch.cuda.is_available() else 'cpu');torch.set_num_threads(4);torch.use_deterministic_algorithms(True);torch.backends.cudnn.benchmark=False
random.seed(42);np.random.seed(42);torch.manual_seed(42)
def sha(p):
 h=hashlib.sha256()
 with p.open('rb') as f:
  for b in iter(lambda:f.read(4194304),b''):h.update(b)
 return h.hexdigest()
class TabularBiLSTM(nn.Module):
 def __init__(self):
  super().__init__();self.lstm=nn.LSTM(7,32,num_layers=2,batch_first=True,bidirectional=True,dropout=.2);self.shared=nn.Sequential(nn.Dropout(.2),nn.Linear(64,32),nn.SiLU());self.detect=nn.Linear(32,1);self.mag=nn.Linear(32,1)
 def forward(self,x):
  z,_=self.lstm(x);z=self.shared(z.mean(1));return self.detect(z)[:,0],self.mag(z)[:,0]
def load(split,h):
 with h5py.File(DATA,'r') as f:
  g=f[split];keep=g[f'classification_mask_{h}s'][:].astype(bool);reg=g[f'magnitude_mask_{h}s'][:].astype(bool);y=g['labels'][:].astype('float32');m=g['magnitude'][:]
  assert np.array_equal(reg,keep&(y==1)) and np.all(m[y==0]==-1) and np.all(keep[y==0]),'Critical mask/sentinel issue'
  x=g[f'{h}s/X'][:][keep];assert x.shape==(int(keep.sum()),10+10*h,7) and np.isfinite(x).all()
  return dict(x=torch.from_numpy(x),y=torch.from_numpy(y[keep]),m=torch.from_numpy(m[keep]),reg=torch.from_numpy(reg[keep]),rows=np.flatnonzero(keep),common=g['common_cohort_mask'][:][keep].astype(bool),source=g['source_id'].asstr()[:][keep],parent=g['parent_id'].asstr()[:][keep],excluded=int((~keep).sum()))
def batches(d,shuffle=False):
 return DataLoader(TensorDataset(d['x'],d['y'],d['m'],d['reg']),batch_size=256,shuffle=shuffle,num_workers=0,generator=torch.Generator().manual_seed(42))
def loss_parts(a,b,y,m,r,center,scale):
 dl=nn.functional.binary_cross_entropy_with_logits(a,y,reduction='sum');ml=nn.functional.huber_loss(b[r],(m[r]-center)/scale,reduction='sum',delta=1.) if r.any() else b.sum()*0
 return dl/len(y)+ml/r.sum().clamp_min(1),dl,ml
# Sentinel poisoning and empty-regression guard: no noise magnitude can enter loss.
y=torch.tensor([1.,0.]);r=y.bool();m=torch.tensor([4.,-1.]);poison=m.clone();poison[1]=1e9
assert torch.equal(loss_parts(torch.zeros(2),torch.zeros(2),y,m,r,4.,1.)[0],loss_parts(torch.zeros(2),torch.zeros(2),y,poison,r,4.,1.)[0])
assert torch.isfinite(loss_parts(torch.zeros(2),torch.zeros(2),torch.zeros(2),m,torch.zeros(2,dtype=torch.bool),4.,1.)[0])
def epoch(model,loader,center,scale,opt=None):
 model.train(opt is not None);ds=ms=0.;n=ne=0
 with torch.set_grad_enabled(opt is not None):
  for x,y,m,r in loader:
   x,y,m,r=[v.to(DEV) for v in [x,y,m,r]]
   if opt is not None:opt.zero_grad(set_to_none=True)
   a,b=model(x);loss,dl,ml=loss_parts(a,b,y,m,r,center,scale);assert torch.isfinite(loss),'Nonfinite loss: stop'
   if opt is not None:loss.backward();nn.utils.clip_grad_norm_(model.parameters(),5.);opt.step()
   ds+=dl.item();ms+=ml.item();n+=len(y);ne+=int(r.sum())
 return dict(detection=ds/n,magnitude=ms/max(ne,1),total=ds/n+ms/max(ne,1))
@torch.inference_mode()
def infer(model,x):
 model.eval();a=[];b=[]
 for chunk in x.split(256):
  l,m=model(chunk.to(DEV));a.append(l.cpu());b.append(m.cpu())
 return torch.cat(a).sigmoid().numpy(),torch.cat(b).numpy()
def detection(y,p,t):
 b=p>=t
 return dict(accuracy=float(accuracy_score(y,b)),precision=float(precision_score(y,b,zero_division=0)),recall=float(recall_score(y,b,zero_division=0)),f1=float(f1_score(y,b,zero_division=0)),roc_auc=float(roc_auc_score(y,p)),pr_auc_ap=float(average_precision_score(y,p)),confusion_matrix=confusion_matrix(y,b,labels=[0,1]).tolist())
def magnitude(a,p):
 a=np.asarray(a,dtype=float);p=np.asarray(p,dtype=float);e=p-a
 return dict(n=len(a),mae=float(np.abs(e).mean()),rmse=float(np.sqrt(np.mean(e**2))),r2=float(r2_score(a,p)),bias=float(e.mean()),actual_variance=float(a.var()),predicted_variance=float(p.var()),std_ratio=float(p.std()/a.std()),slope=float(np.polyfit(a,p,1)[0]),actual_range=[float(a.min()),float(a.max())],predicted_range=[float(p.min()),float(p.max())])
print('Device',str(DEV),'parameters',sum(v.numel() for v in TabularBiLSTM().parameters()))


Device cuda parameters 37730


In [2]:
protected={str(p):sha(p) for p in [DATA,VALIDATION,CNN]}
assert not any((OUT/'models').iterdir()),'Existing checkpoints: refuse overwrite/retraining'
with h5py.File(DATA,'r') as f:
 for h in [1,2,3]:assert f[f'scalers/{h}s'].attrs['fit_split']=='train'
scaler_metadata={}
with h5py.File(DATA,'r') as f:
 for h in [1,2,3]:scaler_metadata[h]={'mean':f[f'scalers/{h}s/mean'][:].tolist(),'scale':f[f'scalers/{h}s/scale'][:].tolist(),'fit_records':int(f[f'scalers/{h}s'].attrs['fit_records'])}
for h in [1,2,3]:
 random.seed(42);np.random.seed(42);torch.manual_seed(42)
 if DEV.type=='cuda':torch.cuda.manual_seed_all(42)
 tr=load('train',h);va=load('validation',h);center=float(tr['m'][tr['reg']].mean());scale=float(tr['m'][tr['reg']].std(unbiased=False));assert scale>0
 model=TabularBiLSTM().to(DEV);opt=torch.optim.AdamW(model.parameters(),lr=.001,weight_decay=.0001);tl=batches(tr,True);te=batches(tr);vl=batches(va)
 history=[];best=float('inf');stale=0;state=None;start=time.perf_counter()
 for e in range(1,36):
  epoch(model,tl,center,scale,opt);a=epoch(model,te,center,scale);b=epoch(model,vl,center,scale);history.append(dict(epoch=e,train=a,validation=b))
  if b['total']<best-.0001:best=b['total'];bestepoch=e;stale=0;state={k:v.detach().cpu().clone() for k,v in model.state_dict().items()}
  else:stale+=1
  if e==1 or e%10==0:print(f'{h}s epoch {e}: train {a["total"]:.4f}, validation {b["total"]:.4f}',flush=True)
  if stale>=7:break
 model.load_state_dict(state);vp,vz=infer(model,va['x']);vy=va['y'].numpy()
 # Prespecified grid; validation only. Ties prefer closest to .5, then lower threshold.
 candidates=np.round(np.arange(.10,.901,.01),2);scores=[float(f1_score(vy,vp>=t,zero_division=0)) for t in candidates];j=max(range(len(candidates)),key=lambda i:(scores[i],-abs(candidates[i]-.5),-candidates[i]));threshold=float(candidates[j])
 ck=dict(state_dict=state,horizon=h,config=CFG,center=center,scale=scale,threshold=threshold,threshold_rule='validation F1 grid .10:.01:.90; tie closest .5 then lower',validation_detection=detection(vy,vp,threshold),validation_magnitude=magnitude(va['m'][va['reg']].numpy(),(vz*scale+center)[va['reg'].numpy()]),best_epoch=bestepoch,best_validation_loss=best,history=history,training_seconds=time.perf_counter()-start,feature_scaler=scaler_metadata[h],input_hashes=protected,counts={s:{'eligible':len(d['y']),'earthquakes':int(d['reg'].sum()),'excluded':d['excluded']} for s,d in [('train',tr),('validation',va)]},parameters=sum(v.numel() for v in model.parameters()),environment={'torch':str(torch.__version__),'device':str(DEV),'hardware':torch.cuda.get_device_name(0) if DEV.type=='cuda' else 'CPU'})
 with (OUT/'models'/f'bilstm_{h}s_best.pt').open('xb') as f:torch.save(ck,f)
 print(f'{h}s selected epoch {bestepoch}/{e}; threshold {threshold:.2f}; fit {ck["training_seconds"]:.1f}s',flush=True)
assert all(sha(Path(p))==v for p,v in protected.items())
print('All models and thresholds selected using validation only; no test predictions yet.')


1s epoch 1: train 1.0416, validation 1.0424
1s epoch 10: train 0.9595, validation 0.9615
1s epoch 20: train 0.9552, validation 0.9575
1s selected epoch 18/25; threshold 0.47; fit 22.0s
2s epoch 1: train 1.0350, validation 1.0430
2s epoch 10: train 0.8670, validation 0.8794
2s epoch 20: train 0.8678, validation 0.8789
2s epoch 30: train 0.8532, validation 0.8698
2s selected epoch 27/34; threshold 0.36; fit 28.5s
3s epoch 1: train 1.0159, validation 1.0299
3s epoch 10: train 0.8174, validation 0.8468
3s epoch 20: train 0.7974, validation 0.8302
3s epoch 30: train 0.7828, validation 0.8202
3s selected epoch 26/33; threshold 0.36; fit 27.9s
All models and thresholds selected using validation only; no test predictions yet.
